Import neccessary libraries

In [41]:
import os
import json
import yaml
from pathlib import Path
import logging

import pandas as pd
import optuna
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score
)

# 1. THIẾT LẬP LOGGING (Dùng force=True để Jupyter không bị lỗi khi chạy lại cell nhiều lần)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True 
)
logger = logging.getLogger(__name__)

# 2. ĐƯỜNG DẪN DỮ LIỆU
CLEANED_DATA_PATH = Path("../data/processed/cleaned_data.csv")
CONFIG_PATH = Path("../artifacts/best_model_config.yaml")
TARGET_COLUMN = "churn_status"

In [22]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Netflix_All_Models_Tuning")
mlflow.sklearn.autolog()

overall_results = {}

2026/04/05 22:17:25 INFO mlflow.tracking.fluent: Experiment with name 'Netflix_All_Models_Tuning' does not exist. Creating a new experiment.


In [23]:
df = pd.read_csv(CLEANED_DATA_PATH)

In [24]:
train_set = df[df['data_split'] == 'train']
test_set = df[df['data_split'] == 'test']

X_train = train_set.drop(columns=['churn_status', 'data_split'])
y_train = train_set['churn_status']

X_test = test_set.drop(columns=['churn_status', 'data_split'])
y_test = test_set['churn_status']

### RANDOM FOREST

In [42]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500), 
        "max_depth": trial.suggest_int("max_depth", 5, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "random_state": 42,
        "class_weight": "balanced",
        "n_jobs": -1 
    }

    with mlflow.start_run(nested=True):
        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        metrics = {
            "pr_auc": average_precision_score(y_test, y_proba),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "log_loss": log_loss(y_test, y_proba),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred)
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        
        return metrics["pr_auc"]

with mlflow.start_run(run_name="RandomForest_PR_AUC_Optimization"):
    study_rf = optuna.create_study(direction="maximize")
    study_rf.optimize(objective, n_trials=10) 

    mlflow.log_params(study_rf.best_params)
    mlflow.log_metric("best_pr_auc", study_rf.best_value)

    best_rf = RandomForestClassifier(**study_rf.best_params, random_state=42)
    best_rf.fit(X_train, y_train)
    
    mlflow.sklearn.log_model(best_rf, "best_rf_prauc_model")

    overall_results["RandomForest"] = {
        "model": best_rf,
        "pr_auc": study_rf.best_value,
        "params": study_rf.best_params
    }

    print("-" * 30)
    print(f"RandomForest Xong! Best PR-AUC: {study_rf.best_value:.4f}")
    print(f"Best Parameters: {study_rf.best_params}")

[I 2026-04-05 23:53:13,866] A new study created in memory with name: no-name-306c6c6b-3fda-49d7-8d0d-36d1b9e412bc
2026/04/05 23:53:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-04-05 23:53:37,281] Trial 0 finished with value: 0.240554531496947 and parameters: {'n_estimators': 223, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 5, 'criterion': 'entropy'}. Best is trial 0 with value: 0.240554531496947.


🏃 View run ambitious-boar-463 at: http://127.0.0.1:5000/#/experiments/4/runs/dd9628cf53944633ba74facbe32ac37a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/05 23:53:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-04-05 23:53:58,296] Trial 1 finished with value: 0.23967847492926375 and parameters: {'n_estimators': 138, 'max_depth': 14, 'min_samples_split': 7, 'min_samples_leaf': 4, 'criterion': 'entropy'}. Best is trial 0 with value: 0.240554531496947.


🏃 View run powerful-doe-223 at: http://127.0.0.1:5000/#/experiments/4/runs/c4542a34171e4036be70d3407174ebb9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/05 23:54:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[W 2026-04-05 23:54:39,644] Trial 2 failed with parameters: {'n_estimators': 227, 'max_depth': 18, 'min_samples_split': 9, 'min_samples_leaf': 1, 'criterion': 'entropy'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "e:\Documents\DSEB.NEU.6TH\ML_OPS\MLOpsProject\MLOpsProject\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Ms Nhan\AppData\Local\Temp\ipykernel_29776\1577914117.py", line 15, in objective
    model.fit(X_train, y_train)
  File

🏃 View run fortunate-cod-301 at: http://127.0.0.1:5000/#/experiments/4/runs/3a7c452d48444d9691a090d9650a0bc6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run RandomForest_PR_AUC_Optimization at: http://127.0.0.1:5000/#/experiments/4/runs/f04a23de551d4955a151b6ce5efd2286
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


KeyboardInterrupt: 

In [26]:
# import matplotlib.pyplot as plt
# import pandas as pd

# # 1. Lấy độ quan trọng của các feature từ model tốt nhất
# importances = best_model.feature_importances_
# feature_names = X_train.columns

# # 2. Tạo DataFrame để dễ sắp xếp
# feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# # 3. In ra Top 5
# print("Top 5 Important Features:")
# print(feature_importance_df.head(5))

# # 4. Vẽ biểu đồ trực quan
# plt.figure(figsize=(10, 6))
# plt.barh(feature_importance_df['Feature'].head(5), feature_importance_df['Importance'].head(5), color='skyblue')
# plt.xlabel('Importance Score')
# plt.title('Top 5 Features causing potential Data Leakage')
# plt.gca().invert_yaxis()
# plt.show()

### XGBOOST

In [27]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 5, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, ratio * 1.5),
        "random_state": 42,
        "eval_metric": "logloss"
    }

    with mlflow.start_run(nested=True):
        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        metrics = {
            "pr_auc": average_precision_score(y_test, y_proba),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "log_loss": log_loss(y_test, y_proba),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred)
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        
        return metrics["pr_auc"]

with mlflow.start_run(run_name="XGBoost_PR_AUC_Optimization"):
    study_xgb = optuna.create_study(direction="maximize")
    study_xgb.optimize(objective, n_trials=50)

    mlflow.log_params(study_xgb.best_params)
    mlflow.log_metric("best_pr_auc", study_xgb.best_value)

    best_xgb = XGBClassifier(**study_xgb.best_params, random_state=42)
    best_xgb.fit(X_train, y_train)
    
    mlflow.xgboost.log_model(best_xgb, "best_xgb_prauc_model")

    overall_results["XGBoost"] = {
        "model": best_xgb,
        "pr_auc": study_xgb.best_value,
        "params": study_xgb.best_params
    }

    print("-" * 30)
    print(f"XGBoost Xong! Best PR-AUC: {study_xgb.best_value:.4f}")
    print(f"Best Parameters: {study_xgb.best_params}")

[I 2026-04-05 22:23:12,475] A new study created in memory with name: no-name-d5d3eb17-448e-47a5-a9b6-1cdf76ae5540
[I 2026-04-05 22:23:15,185] Trial 0 finished with value: 0.23313312389602578 and parameters: {'n_estimators': 862, 'max_depth': 14, 'learning_rate': 0.04532022027942535, 'subsample': 0.7471198089426614, 'colsample_bytree': 0.8150837605427215, 'gamma': 3.7669688983200977, 'scale_pos_weight': 1.362034643935202}. Best is trial 0 with value: 0.23313312389602578.


🏃 View run traveling-fowl-201 at: http://127.0.0.1:5000/#/experiments/4/runs/d58ec06eca6742979f84d7c6e2a61710
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-05 22:23:17,776] Trial 1 finished with value: 0.21069015974628613 and parameters: {'n_estimators': 552, 'max_depth': 7, 'learning_rate': 0.138974850902833, 'subsample': 0.9720773100591744, 'colsample_bytree': 0.6412582232459212, 'gamma': 0.11443750076362336, 'scale_pos_weight': 1.884597760153628}. Best is trial 0 with value: 0.23313312389602578.


🏃 View run victorious-smelt-59 at: http://127.0.0.1:5000/#/experiments/4/runs/de6011f1e2214be5ad1fe481c348ab45
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-05 22:23:18,745] Trial 2 finished with value: 0.23891777768858213 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.010730771654905614, 'subsample': 0.5766036947977473, 'colsample_bytree': 0.6241938561390175, 'gamma': 4.930574262912452, 'scale_pos_weight': 6.665121496822722}. Best is trial 2 with value: 0.23891777768858213.


...

🏃 View run carefree-stork-572 at: http://127.0.0.1:5000/#/experiments/4/runs/db5e9bb44c1d4b09a8be371816f3c0c1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-05 22:25:12,592] Trial 49 finished with value: 0.2387180780286059 and parameters: {'n_estimators': 281, 'max_depth': 8, 'learning_rate': 0.022578596090894416, 'subsample': 0.7234794819498401, 'colsample_bytree': 0.6472268184816478, 'gamma': 3.8639505142804715, 'scale_pos_weight': 2.7763974076081666}. Best is trial 12 with value: 0.24123610791132755.


🏃 View run gifted-gnat-940 at: http://127.0.0.1:5000/#/experiments/4/runs/d46636dc9a94405698919b5608b5c220
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/05 22:25:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


------------------------------
XGBoost Xong! Best PR-AUC: 0.2412
Best Parameters: {'n_estimators': 211, 'max_depth': 9, 'learning_rate': 0.02113376337742602, 'subsample': 0.6221376784929888, 'colsample_bytree': 0.5388734283612984, 'gamma': 4.859984208589209, 'scale_pos_weight': 5.6377853135394}
🏃 View run XGBoost_PR_AUC_Optimization at: http://127.0.0.1:5000/#/experiments/4/runs/32d8d56fa2cd4be6b7800f13894c1fe3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [28]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [29]:
# # Vẽ ma trận nhầm lẫn chuẩn hóa (normalized) theo hàng ngang (true labels)
# fig, ax = plt.subplots(figsize=(8, 6))
# disp_norm = ConfusionMatrixDisplay.from_estimator(
#     best_xgb, 
#     X_test, 
#     y_test, 
#     display_labels=["Active (0)", "Churn (1)"],
#     cmap="Greens",
#     normalize='true', # Quan trọng: tính % theo từng lớp thực tế
#     ax=ax
# )

# plt.title("Normalized Confusion Matrix (%)")
# plt.show()

### LIGHTGBM

In [30]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, ratio * 1.5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "verbosity": -1
    }

    with mlflow.start_run(nested=True):
        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        metrics = {
            "pr_auc": average_precision_score(y_test, y_proba),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "log_loss": log_loss(y_test, y_proba),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred)
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        
        return metrics["pr_auc"]

with mlflow.start_run(run_name="LGBM_PR_AUC_Optimization"):
    study_lgbm = optuna.create_study(direction="maximize")
    study_lgbm.optimize(objective, n_trials=50)

    mlflow.log_params(study_lgbm.best_params)
    mlflow.log_metric("best_pr_auc", study_lgbm.best_value)

    best_lgbm = LGBMClassifier(**study_lgbm.best_params, random_state=42)
    best_lgbm.fit(X_train, y_train)
    
    mlflow.lightgbm.log_model(best_lgbm, "best_lgbm_prauc_model")

    overall_results["LightGBM"] = {
        "model": best_lgbm,
        "pr_auc": study_lgbm.best_value,
        "params": study_lgbm.best_params
    }

    print("-" * 30)
    print(f"LGBM Xong! Best PR-AUC: {study_lgbm.best_value:.4f}")
    print(f"Best Parameters: {study_lgbm.best_params}")

[I 2026-04-05 22:26:01,571] A new study created in memory with name: no-name-a57dc83a-5aa9-4c23-acae-97e69414e32c
[I 2026-04-05 22:26:09,180] Trial 0 finished with value: 0.2397048849521965 and parameters: {'n_estimators': 994, 'learning_rate': 0.0097039608890122, 'num_leaves': 79, 'max_depth': 4, 'min_child_samples': 64, 'scale_pos_weight': 5.4966583605157, 'reg_alpha': 0.02756849148050542, 'reg_lambda': 2.7279334758345892}. Best is trial 0 with value: 0.2397048849521965.


🏃 View run unequaled-ray-267 at: http://127.0.0.1:5000/#/experiments/4/runs/f3c38e82be554e61855b3573fe511e31
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-05 22:26:50,985] Trial 1 finished with value: 0.23003641314930803 and parameters: {'n_estimators': 1077, 'learning_rate': 0.008877070155153998, 'num_leaves': 56, 'max_depth': 7, 'min_child_samples': 66, 'scale_pos_weight': 3.550952953016262, 'reg_alpha': 3.1223123170245635, 'reg_lambda': 0.738224844518938}. Best is trial 0 with value: 0.2397048849521965.


🏃 View run legendary-lamb-589 at: http://127.0.0.1:5000/#/experiments/4/runs/c7ab81e86b0145f38559cafdfb5e50e6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-05 22:27:06,154] Trial 2 finished with value: 0.23704987368283892 and parameters: {'n_estimators': 327, 'learning_rate': 0.006033464103573308, 'num_leaves': 117, 'max_depth': 9, 'min_child_samples': 84, 'scale_pos_weight': 1.052765074354581, 'reg_alpha': 4.958786944155619, 'reg_lambda': 3.284841125763266}. Best is trial 0 with value: 0.2397048849521965.


...

🏃 View run likeable-panda-139 at: http://127.0.0.1:5000/#/experiments/4/runs/f03e6061f87a43968b3e92e349c949f3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-05 22:39:24,544] Trial 49 finished with value: 0.23431065450373922 and parameters: {'n_estimators': 474, 'learning_rate': 0.02367778570622209, 'num_leaves': 111, 'max_depth': 5, 'min_child_samples': 41, 'scale_pos_weight': 4.353747307492484, 'reg_alpha': 0.7545531404285132, 'reg_lambda': 0.04585027945190727}. Best is trial 36 with value: 0.23996170813992868.


🏃 View run blushing-shrew-128 at: http://127.0.0.1:5000/#/experiments/4/runs/3b33f839f7f942f896682d52429d330c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/05 22:39:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 22:39:26 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


------------------------------
LGBM Xong! Best PR-AUC: 0.2400
Best Parameters: {'n_estimators': 633, 'learning_rate': 0.010107135477443534, 'num_leaves': 113, 'max_depth': 6, 'min_child_samples': 46, 'scale_pos_weight': 4.712763656112233, 'reg_alpha': 0.003325524310919295, 'reg_lambda': 0.0019387295682310396}
🏃 View run LGBM_PR_AUC_Optimization at: http://127.0.0.1:5000/#/experiments/4/runs/05819733efca4edaaf100884b6153d8b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [40]:

# 1. Tìm model có điểm pr_auc cao nhất trong cái giỏ "all_models_results"
best_model_name = max(overall_results, key=lambda x: overall_results[x]["pr_auc"])
best_info = overall_results[best_model_name]

print(f"🏆 CHUNG CUỘC - Model xuất sắc nhất: {best_model_name}")
print(f"   PR-AUC: {best_info['pr_auc']:.4f}")

# 2. Tạo Dictionary chứa cấu hình (Chỉ gồm Text và Số)
model_config = {
    'project': 'Netflix_Prediction',
    'best_model_overall': best_model_name,
    'metrics': {
        'pr_auc': best_info['pr_auc']
    },
    'hyperparameters': best_info['params']
}

# Lệnh này cực kỳ quan trọng: Nó giúp tự động tạo thư mục "artifacts" nếu thư mục này chưa tồn tại
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

# Lưu dictionary thành file JSON (dùng indent=4 để file được format đẹp, dễ đọc)
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=4)

print(f"📁 Đã lưu file config ra máy tại: {CONFIG_PATH}")
# ==========================================


# 3. Tạo 1 Run MLflow cuối cùng để chốt sổ dự án (Như cũ)
with mlflow.start_run(run_name=" SUMMARY_FINAL_RESULT"):
    
    mlflow.log_metric("final_best_pr_auc", best_info['pr_auc'])
    mlflow.set_tag("winner_model", best_model_name)
    
    # Bạn vẫn có thể lưu cái Dict này lên web MLflow dưới dạng JSON luôn cho đồng bộ
    mlflow.log_dict(model_config, "configs/final_best_model_config.yaml")
    
    if best_model_name == "LightGBM":
        mlflow.lightgbm.log_model(best_info['model'], "deploy_model")
    elif best_model_name == "XGBoost":
        mlflow.xgboost.log_model(best_info['model'], "deploy_model")

print("✅ Đã lưu toàn bộ lên MLflow Server!")

🏆 CHUNG CUỘC - Model xuất sắc nhất: RandomForest
   PR-AUC: 0.2426
📁 Đã lưu file config ra máy tại: ..\artifacts\best_model_config.yaml
🏃 View run  SUMMARY_FINAL_RESULT at: http://127.0.0.1:5000/#/experiments/4/runs/c3baac884f3e46a2a0e0ce5f8a996697
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
✅ Đã lưu toàn bộ lên MLflow Server!
